In [ ]:
import numpy as np
from scipy.special import digamma
from scipy.stats import beta
import matplotlib.pyplot as plt
from bad import BetaDistr


def bald(a, b):
    mu =  BetaDistr(a, b).mu()
    H_y = -mu * np.log(mu) - (1 - mu) * np.log(1 - mu)
    H_y_given_w = digamma(a + b + 1) - mu * digamma(a + 1) - (1 - mu) * digamma(b + 1)
    return H_y - H_y_given_w


def margin(a, b, r=0.5):
    mode = BetaDistr(a, b).mu()
    logmargin= beta.logpdf(mode, a, b) - beta.logpdf(r, a, b)
    return np.exp(-logmargin)


def anom(a, b):
    mu = BetaDistr(a, b).mu()
    return mu


a = 10 ** np.linspace(-2, 2, 1000)
b = 10 ** np.linspace(-2, 2, 1000)
A, B = np.meshgrid(a, b)

plt.figure(figsize=(15, 4))
for i, fn in enumerate([bald, margin, anom]):
    plt.subplot(1, 3, i + 1)
    plt.contourf(A, B, fn(A, B), levels=20, cmap='viridis')
    plt.colorbar()
    plt.title(fn.__name__)
    plt.xlabel("$\\alpha$")
    plt.ylabel("$\\beta$")
    plt.xscale("log")
    plt.yscale("log")
    plt.gca().set_aspect('equal', adjustable='box')
plt.tight_layout()
plt.savefig("bald_margin_anom.pdf")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from pyod.utils.data import generate_data
from pyod.utils.example import visualize

from iforest import BAD_IForest
from pyod.models.iforest import IForest

X_train, X_test, y_train, y_test = generate_data(
    n_train=1000, n_test=100, contamination=0.1, random_state=0
)
X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)
bforest = BAD_IForest(interest_method="bald").fit(X_train)
iforest = IForest().fit(X_train)

In [ ]:
bforest = BAD_IForest().fit(X_train)
iforest = IForest().fit(X_train)
plt.plot(iforest.decision_function(X_test), label="IForest")
plt.plot(bforest.decision_function(X_test), label="BAD_IForest")
plt.legend()
plt.grid()
plt.show()

In [ ]:
bforest = BAD_IForest().fit(X_train)
plt.plot(bforest.decision_function(X_test), label="BAD_IForest")
plt.hlines(bforest.threshold_, 0, len(X_test), "k", label="Threshold")

queriable = np.ones(len(X_test), dtype=bool)
for i in range(10):
    scores = bforest.interest(X_test)
    i = np.argmax(np.where(queriable, scores, -np.inf))
    bforest.update(X_test[i:i+1], y_test[i:i+1])
    queriable[i] = False

    plt.vlines(i, 0, 1, "r", alpha=0.2)
plt.plot(bforest.decision_function(X_test), label="BAD_IForest+queries")
plt.legend()
plt.grid()
plt.show()

In [ ]:
bforest = BAD_IForest().fit(X_train)
plt.plot(bforest.decision_function(X_test), label="BAD_IForest")
plt.hlines(bforest.threshold_, 0, len(X_test), "k", label="Threshold")
scores = bforest.interest(X_test)
i = np.argsort(scores)[-10:]
bforest.update(X_test[i], y_test[i])

plt.vlines(i, 0, 1, "r", alpha=0.2)
plt.plot(bforest.decision_function(X_test), label="BAD_IForest+queries")
plt.legend()
plt.grid()
plt.show()

In [ ]:
print("IForest fit time:")
%timeit IForest().fit(X_train)
print("IForest predict time:")
%timeit iforest.decision_function(X_test)

In [ ]:
print("BAD_IForest fit time:")
%timeit BAD_IForest(reprocess_decision_scores=False).fit(X_train)
print("BAD_IForest predict time:")
%timeit bforest.decision_function(X_train)

print("BAD_IForest interest time:")
%timeit bforest.interest(X_train)
print("BAD_IForest update time:")
%timeit bforest.update(X_train, y_train)
print("BAD_IForest batch_query time:")
%timeit bforest.get_queries(X_train, 10)